# 00 · Окружение DataSphere: репозиторий, железо, vLLM, смоук на logprobs

Первый ноутбук любой GPU-сессии. Ставит стек, клонирует **рабочую ветку**, поднимает
vLLM и проверяет, что извлечение вероятностей из logprobs действительно работает.

**Конфигурация: `g2.1` (1× A100 80 GB).** Всё, что связано с LLM-инференсом или
обучением 7B, требует её — см. `docs/specs/90_DATASPHERE_runbook.md` §1.

Ноутбук — пусковая установка: вся логика живёт в репозитории, ячейки только
поднимают окружение и вызывают CLI. Это и сохраняет `run.yaml` с git-хэшем рядом
с каждым прогоном.

Порядок: **Run All**, затем перейти к `10_score_judge.ipynb` / `20_train_encoder.ipynb`.

## 1. Конфигурация и репозиторий

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE     = "/home/jupyter/filestore/neurodrive"   # File Storage: переживает рестарт VM
REPO     = f"{BASE}/rag-reliability"
REPO_URL = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH   = "integration"                          # НЕ qwen7b-notebook: та ветка устарела
CACHE    = f"{BASE}/cache/m3_judge"               # кэш судьи -> прогон резюмируется
LOGS     = f"{BASE}/logs"
DATA     = "data/alfa.jsonl"                      # канонический корпус, 2233 кейса
FOLDS    = "data/splits/folds_alfa.json"          # сплит только отсюда, split_samples не вызываем
MODEL    = "Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
# ===================================================================================

import os, subprocess, sys

# HF_HOME до любого импорта transformers: кэш модели ~15 GB не влезает в проектный диск.
os.environ["HF_HOME"] = f"{BASE}/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("OPENAI_API_KEY", "dummy")
for directory in (LOGS, CACHE, f"{BASE}/hf"):
    os.makedirs(directory, exist_ok=True)

if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "pull", "--ff-only"])

# Коммит артефактов прямо из ноутбука невозможен без идентичности.
subprocess.check_call(["git", "-C", REPO, "config", "user.name", "datasphere-runner"])
subprocess.check_call(["git", "-C", REPO, "config", "user.email", "datasphere@localhost"])

head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).decode().strip()
branch = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]
).decode().strip()
print(f"repo {REPO} | branch {branch} | HEAD {head}")
assert branch == BRANCH, f"ожидалась ветка {BRANCH}, а рабочее дерево на {branch}"


## 2. Стек

Драйвер DataSphere — CUDA 12.2. `torch` с PyPI (cu13) падает «driver too old», базовый
`torch 2.0.1/cu118` слишком стар для актуального `trl`. Отсюда пин `torch==2.5.1` на cu121;
`vllm` подобран под ровно эту версию torch — менять одно без другого нельзя.
`numpy==1.26.4` ставится **последним**: numpy 2 ломает C-расширения базового образа
(`soxr`, `scipy`, `sklearn`, `numba`).

После первого прогона этой ячейки — **Kernel → Restart** и снова Run All.

In [ ]:
pip = lambda *args: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121")
pip("vllm==0.6.6.post1")                 # собран под torch 2.5.1
pip("-e", f"{REPO}[m6,encoder,gepa,cloud]")   # NLI/эмбеддинги + энкодер + DSPy + клиент судьи
pip("numpy==1.26.4")                     # ПОСЛЕДНИМ: перекрывает numpy 2, если его подтянули выше
print("стек установлен; если torch уже импортировался в этом ядре — Kernel → Restart и Run All")


## 3. Железо

In [ ]:
import torch, psutil

n = torch.cuda.device_count()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if n else 0.0
ram = psutil.virtual_memory().total / 1e9
print(f"GPU {torch.cuda.get_device_name(0) if n else '—'} | VRAM {vram:.0f}GB | "
      f"RAM {ram:.0f}GB | GPUs {n}")

assert n >= 1, (
    "выбрана CPU-конфигурация. Судья, энкодер на 8192 токенах и FT 7B требуют GPU: "
    "Ресурсы проекта → Конфигурация вычислительных ресурсов → g2.1 → рестарт VM"
)
assert vram >= 70, (
    f"VRAM {vram:.0f} GB < 70 GB. Этот контур рассчитан на g2.1 (1× A100 80 GB): "
    "на g1.1 (V100 32 GB) нет bf16, а flash-attention 2 требует Ampere. "
    "Меньшие конфигурации годятся только для NLI-grounding (gt4i.1) — "
    "см. docs/specs/90_DATASPHERE_runbook.md §1"
)


## 4. Хранилище и сплит

`folds.json` — единственный источник разбиения. `split_samples` в этом контуре не
вызывается нигде: он давал утечку 24.9% по вопросу, и полученные им числа несравнимы
с числами на group-сплитах.

In [ ]:
for relative in (DATA, FOLDS):
    path = f"{REPO}/{relative}"
    assert os.path.isfile(path), (
        f"нет {path}. Либо File Storage не смонтирован (Ресурсы проекта → Файловое "
        f"хранилище → активировать → рестарт VM), либо клонирована не та ветка (нужна {BRANCH})"
    )
print("данные и фолды на месте")


In [ ]:
!cd {REPO} && python scripts/prepare_splits.py --check --data {DATA} --folds {FOLDS}

## 5. vLLM в фоне

`--max-logprobs 25` обязателен: клиент судьи запрашивает `top_logprobs=20`, и без
этого флага сервер вернёт вердикт без вероятностей — вся ветка Метода 3 деградирует
в regex-парсинг.

Лог пишется на File Storage, а не в `/tmp`: `/tmp` не переживает рестарт VM.

In [ ]:
import time, requests

subprocess.Popen(
    f"vllm serve {MODEL} --port 8000 --max-model-len 8192 "
    f"--gpu-memory-utilization 0.85 --max-logprobs 25 "
    f"> {LOGS}/vllm.log 2>&1",
    shell=True,
)

for _ in range(120):                    # до 10 минут: первая загрузка весов долгая
    try:
        if requests.get(f"{API_BASE}/models", timeout=2).ok:
            print("vLLM up")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError(
        f"vLLM не поднялся за 10 минут — смотри {LOGS}/vllm.log. Типовая причина: "
        "не хватило VRAM под KV-кэш; понизить --gpu-memory-utilization до 0.75 "
        "и --max-model-len до 4096"
    )


## 6. Преполётная проверка CLI

Ноутбуки ничего не считают сами — они вызывают CLI. Если точки входа нет, это баг CLI,
а не повод писать расчёт в ячейке. Здесь видно, что доступно на текущей ветке.

In [ ]:
CLI_NEEDED = {
    "scripts/score.py": "прогоны судьи по корпусу с --resume (B2)",
    "scripts/run_m3.py": "разделённые оси и self-consistency (C3)",
    "scripts/evaluate_cv.py": "оценка 5x5 CV с ДИ (B1)",
    "scripts/train_encoder_baseline.py": "OOF-обучение энкодера (C2)",
    "scripts/run_gepa.py": "эволюция промпта, запускается через Jobs (D2)",
    "scripts/train_ft_judge.py": "FT судьи по фолдам (нет в репозитории — см. PR D3)",
}
for relative, why in CLI_NEEDED.items():
    mark = "OK " if os.path.isfile(f"{REPO}/{relative}") else "НЕТ"
    print(f"{mark}  {relative:36s} {why}")


## 7. Смоук на logprobs — обязательный, не пропускать

Токенизатор vLLM отличается от того, что был у OpenRouter-провайдера, а извлечение
вердикта чувствительно к разбиению `PASS`/`FAIL` на подтокены. При односимвольном
первом подтокене вероятность **молча** становится 0.5 для всех кейсов — прогон
выглядит успешным, а сигнала в нём нет.

Пять кейсов, ~10 секунд. Красный ассерт здесь означает: полный прогон не запускать.

In [ ]:
!cd {REPO} && python scripts/score.py --method m3_openai_judge --variant zero_shot \
    --data {DATA} --limit 5 --model {MODEL} \
    --m3-api-base {API_BASE} --m3-cache-dir {BASE}/cache/smoke \
    --output {BASE}/smoke/scores.jsonl


In [ ]:
import json

rows = [json.loads(line) for line in open(f"{BASE}/smoke/scores.jsonl", encoding="utf-8")]
print("prob_method:", [row["prob_method"] for row in rows])
print("m3.p_faith :", [round(row["scores"]["m3.p_faith"], 4) for row in rows])

assert rows, "смоук не дал ни одной строки — смотри вывод предыдущей ячейки"
assert all(row["prob_method"] == "logprobs" for row in rows), (
    "PASS/FAIL режется на подтокены: извлечение вырождается в 0.5 для всех кейсов. "
    "Чинить _verdict_positions (задача A4) ДО полного прогона и инвалидировать кэш судьи. "
    "prob_method == 'regex' на всех строках — другой симптом: сервер не отдаёт logprobs, "
    "проверить --max-logprobs 25 у vLLM"
)
assert len({round(row["scores"]["m3.p_faith"], 3) for row in rows}) > 1, (
    "все вероятности одинаковы — извлечение сломано"
)
print("logprobs smoke OK — можно запускать 10_score_judge.ipynb")
